In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
import skimage.io
import time
import torch
import glob
from scipy.spatial import KDTree
from scipy.spatial import Delaunay

import sys
sys.path.append("../src")
from geometry import get_image_curve, estimate_normals_curve, reconstruct_signed_voronoi_2d, get_curv_vectors, get_sdf_2d
from imgvoronoi import stochastic_universal_sample

X = get_image_curve("../data/images/fish.png")
#t = np.linspace(0, 2*np.pi, 1000)
#X = np.array([np.cos(t), np.sin(t)]).T
X = X-np.min(X, axis=0, keepdims=True)
X = X/np.max(np.abs(X))

N = estimate_normals_curve(X)

In [ ]:
Curvs = get_curv_vectors(X, 2, sigma=10)
curv = np.sqrt(np.sum(Curvs[-1]**2, axis=1))
curv = curv**0.4
curv = curv/np.sum(curv)

In [ ]:
n_points = 100
n_iters = 500

idx = np.array(stochastic_universal_sample(curv, n_points), dtype=int)[:, 0]
XSamples = np.array(X[idx, :])

In [ ]:
plt.scatter(X[:, 0], X[:, 1], s=5, c=curv)

In [ ]:
res = 128
anchors = np.meshgrid(np.linspace(1.2, -0.2, res), np.linspace(-0.2, 1.2, res), indexing='ij')
anchors = np.concatenate(tuple([Ai.flatten()[:, None] for Ai in anchors]), axis=1)

tree = KDTree(X)
d, _ = tree.query(anchors, k=1, workers=-1)
d = d.flatten()
anchors = anchors[d < 0.05, :]

sdf = get_sdf_2d(X, N, anchors)
bsdf = np.sign(sdf)
bsdf = 0.5*(1+bsdf)
print(anchors.shape, X.shape)
anchors = np.concatenate((anchors, X), axis=0)
bsdf = np.concatenate((bsdf, 0.5*np.ones(X.shape[0])))

#plt.scatter(X[:, 0], X[:, 1])
plt.scatter(anchors[:, 0], anchors[:, 1], c=bsdf, cmap='RdBu')
plt.colorbar()

In [ ]:
anchors = torch.from_numpy(np.array(anchors))
bsdf = torch.from_numpy(np.array(bsdf))
XSamples  = torch.from_numpy(XSamples).requires_grad_()
c = torch.zeros(XSamples.shape[0])
c[XSamples.shape[0]//2:] = 1

In [ ]:
res = 512
# G is for plotting the entire 2D SDF
G = np.meshgrid(np.linspace(1.2, -0.2, res), np.linspace(-0.2, 1.2, res), indexing='ij')
G = np.concatenate(tuple([Gi.flatten()[:, None] for Gi in G]), axis=1)


optimizer = torch.optim.Adam([XSamples], lr=3e-4)

temperature = 1
losses = []
pidx = 0

fac = 2
plt.figure(figsize=(10*fac, 10*fac))
for it in range(n_iters):
    XSUsing = XSamples.detach().cpu().numpy()
    optimizer.zero_grad()

    ## Compute distance from all pixels to landmarks
    n_neighbs = 2
    tree = KDTree(XSUsing)
    _, nn_idx = tree.query(anchors.detach().cpu().numpy(), k=n_neighbs, workers=-1) # Nearest landmark indices to anchors
    
    ## Based on the landmark locations, compute the weight influence
    ## of each landmark on each pixel using softmax
    weights = torch.zeros(anchors.shape[0], n_neighbs)
    for i in range(n_neighbs):
        dX = XSamples[nn_idx[:, i], :] - anchors
        weights[:, i] = 1/(1e-4+torch.sum(dX**2, dim=1))
    mx = weights[:, 0].view(weights.shape[0], 1)
    weights = torch.exp(temperature*weights/mx)
    weights = weights/torch.sum(weights, dim=1, keepdims=True)

    ## Splat colors onto anchors
    J = torch.zeros(anchors.shape[0])
    for i in range(n_neighbs):
        J += c[nn_idx[:, i]]*weights[:, i]
    all_losses = (J-bsdf)**2

    # Compute loss
    loss = torch.sum(all_losses)
    losses.append(loss.item())
    
    loss.backward()
    optimizer.step()
    
    if it%5 == 0:
        XSUsing = XSamples.detach().cpu().numpy()
        plt.clf()
        plt.subplot(221)
        plt.plot(X[:, 0], X[:, 1], linewidth=1, c='k')
        plt.scatter(XSUsing[:, 0], XSUsing[:, 1], c=c.numpy()[0:XSUsing.shape[0]], s=1, cmap='RdBu')
        plt.axis("equal")
        plt.xlim([-0.2, 1.2])
        plt.ylim([-0.2, 1.2])
        plt.title("Using {} sites".format(n_points*2))
    
        plt.subplot(222)
        tree = KDTree(XSUsing)
        _, idx = tree.query(G)
        idx = c[idx].detach().cpu().numpy()
        plt.imshow(np.fliplr(idx.reshape((res, res)).T), cmap='RdBu', extent=(-0.2, 1.2, -0.2, 1.2), origin='lower')
        plt.title("Binarized SDF")
    
        ## Plot reconstruction so far
        plt.subplot(223)
        centers, edges = reconstruct_signed_voronoi_2d(XSUsing, c.cpu().numpy())
        plt.plot(X[:, 0], X[:, 1], c='k', linestyle='--')
        for [e1, e2] in edges:
            Xi = centers[[e1, e2], :]
            plt.plot(Xi[:, 0], Xi[:, 1])
        plt.title("Reconstruction")
        
        
        plt.subplot(224)
        plt.plot(losses)
        plt.title("Loss {:.5f}".format(loss.item()))
        plt.xlabel("Iteration")
        plt.title("Loss")

        
        plt.savefig("Iter{}.png".format(pidx))
        pidx += 1

